# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
import getpass
import duckdb

# Get the Hugging Face token securely
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

HF_TOKEN = get_hf_token()

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 development partition
FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("Using March 2026 development data.")

DuckDB connection established.
Using March 2026 development data.


### Feature vector

For the March 2026 development window, I will represent each client-content pair using five performance features: impressions, clicks, CTR, average position, and position volatility.

The features are aggregated from the same March 2026 window so that each observation has one consistent feature vector. Client and content IDs are retained only for identification and grouping, not as clustering features.

Rows without available GSC data are not treated as zero performance; GSC-derived metrics are calculated only where the relevant data is available.

In [4]:
# Inspect the actual columns available in the March 2026 fact table

schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM {FACT_DAILY}
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [5]:
# Show all column names in the March 2026 fact table

for column in schema["column_name"]:
    print(column)


report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [6]:
# Build the first five clustering features from March 2026

feature_vector = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

display(feature_vector.head())

print("Number of client-content pairs:", len(feature_vector))
print("Number of clustering features:", 5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,7.209549,2.442255
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.307255,2.396517
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.724039,1.243547
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.244844,0.983020
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,4.499519,4.340230


Number of client-content pairs: 176738
Number of clustering features: 5


### Feature notes

The feature vector contains five performance features calculated from the March 2026 development window.

| Feature | Meaning | Missing-value handling | Available when |
|---|---|---|---|
| `impressions` | Total Google Search impressions for the client-content pair during March 2026. It represents search visibility. | Calculated only from rows where GSC data is available. | Available when GSC data has been recorded. |
| `clicks` | Total Google Search clicks received by the client-content pair during March 2026. | Calculated only from rows where GSC data is available. | Available when GSC data has been recorded. |
| `ctr` | Click-through rate calculated as clicks divided by impressions, multiplied by 100. | NULL when total impressions are zero. | Available after impressions and clicks have been observed. |
| `avg_position` | Mean observed Google Search average position during March 2026. | Position values of 0 are treated as missing before averaging. | Available when valid GSC position data exists. |
| `position_volatility` | Sample standard deviation of daily average position, measuring how much observed position varies during March. | Zero positions are treated as missing; insufficient valid observations can result in NULL. | Available after sufficient valid position observations exist. |

`client_hash_id` and `content_hash_id` are retained for identification and grouping but are excluded from the clustering feature set because they are identifiers rather than performance signals.

All five features are based only on the March 2026 development window. No future-month information is used.

In [7]:
# Check missing values in the five clustering features

clustering_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]

feature_missingness = (
    feature_vector[clustering_features]
    .isna()
    .sum()
    .to_frame("missing_rows")
)

feature_missingness["missing_pct"] = (
    100.0
    * feature_missingness["missing_rows"]
    / len(feature_vector)
)

display(feature_missingness)

,missing_rows,missing_pct
impressions,0,0.000000
clicks,0,0.000000
ctr,0,0.000000
avg_position,1434,0.811371
position_volatility,15181,8.589551


### Leakage test result

The honest March-only feature set produced a silhouette score of 0.5615 on the 20,000-row sample, while adding the future April 2026 impressions produced a silhouette score of 0.5464.

The silhouette score therefore decreased by 0.0151 when the future feature was added. This does not mean future information is safe to use. The April feature is still invalid because it was not available at the March decision point.

The deliberate test confirms the key rule for this project: clustering features must be constructed only from information available within the defined March 2026 development window.

`future_april_impressions` is therefore excluded from the final feature vector.

In [10]:
# Deliberate leakage test:
# add April 2026 impressions, which were not available at the March decision point.

future_april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_april_impressions
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

leakage_frame = feature_vector.merge(
    future_april,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

leakage_frame["future_april_impressions"] = (
    leakage_frame["future_april_impressions"].fillna(0)
)

display(leakage_frame.head())

print("March feature rows:", len(feature_vector))
print("Rows with April information:", (leakage_frame["future_april_impressions"] > 0).sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility,future_april_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,7.209549,2.442255,6787.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.307255,2.396517,405.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.724039,1.243547,8475.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.244844,0.983020,6091.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,4.499519,4.340230,287.0


March feature rows: 176738
Rows with April information: 158549


In [11]:
# Compare honest clustering with clustering that contains future information

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np

# Use the same sample for both experiments so the comparison is fair
sample = leakage_frame.sample(
    n=min(20000, len(leakage_frame)),
    random_state=42
).copy()

# The five legitimate March features
honest_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]

# The deliberately leaky feature
leaky_features = honest_features + [
    "future_april_impressions"
]

def prepare_features(data, columns):
    """
    Select features, fill missing values with the median,
    and standardize the numerical values.
    """
    X = data[columns].copy()

    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(X)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X


# Prepare the honest March-only feature matrix
X_honest = prepare_features(sample, honest_features)

# Prepare the deliberately leaky feature matrix
X_leaky = prepare_features(sample, leaky_features)


# Use the same number of clusters for both experiments
k = 5

honest_model = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

leaky_model = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

honest_labels = honest_model.fit_predict(X_honest)
leaky_labels = leaky_model.fit_predict(X_leaky)


# Compare the clustering quality
honest_silhouette = silhouette_score(
    X_honest,
    honest_labels,
    sample_size=5000,
    random_state=42
)

leaky_silhouette = silhouette_score(
    X_leaky,
    leaky_labels,
    sample_size=5000,
    random_state=42
)


print("Leakage experiment")
print("------------------")
print("Sample size:", len(sample))
print("Number of clusters:", k)
print()
print("Honest March-only silhouette:", round(honest_silhouette, 4))
print("Leaky March + April silhouette:", round(leaky_silhouette, 4))
print(
    "Change in silhouette:",
    round(leaky_silhouette - honest_silhouette, 4)
)

Leakage experiment
------------------
Sample size: 20000
Number of clusters: 5

Honest March-only silhouette: 0.5615
Leaky March + April silhouette: 0.5464
Change in silhouette: -0.0151


### Excluded fields

| Excluded field | Reason |
|---|---|
| `client_hash_id` | Identifier only; retained for grouping and traceability but not used as a clustering feature. |
| `content_hash_id` | Identifier only; retained for grouping and traceability but not used as a clustering feature. |
| `report_date` | Used to define the March 2026 window but not used as a clustering feature. |
| `month` | Redundant with the explicitly selected March 2026 development window. |
| `gsc_sum_position` | Not needed because the warehouse already provides `gsc_avg_position`, which is the more directly interpretable position measure. |
| `gsc_data_available` | Used as a data-quality filter, not as a performance feature. |
| `client_has_gsc` | Data-availability/context field rather than a content-performance signal. |
| `client_has_ga4` | Data-availability/context field rather than a content-performance signal. |
| `ga4_data_available` | Data-availability/context field and not required for the initial GSC-based clustering feature set. |
| GA4 traffic and engagement fields | Excluded from the initial five-feature vector to keep the first clustering feature set focused on comparable search-performance signals. |
| AI referral/session fields | Excluded from the initial feature vector because they are not required for the first GSC-based archetype definition. |
| `future_april_impressions` | Deliberately created for the leakage test but excluded because April 2026 information would not have been available at the March decision point. |
| `is_declining_label` / trend-derived fields | Excluded because they are label-derived and would introduce information derived from an outcome rather than an independent clustering signal. |

In [12]:
# Final feature-set verification

print("Final clustering features:")
for feature in honest_features:
    print(" -", feature)

print("\nExcluded leakage feature:")
print(" - future_april_impressions")

print("\nNumber of final clustering features:", len(honest_features))

Final clustering features:
 - impressions
 - clicks
 - ctr
 - avg_position
 - position_volatility

Excluded leakage feature:
 - future_april_impressions

Number of final clustering features: 5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.